# Text Splitters and Chunking

Chunking breaks long documents into smaller pieces that are easier to embed, retrieve, and pass to a language model. Good chunks preserve useful context and fit your downstream model.

`chunk_size` and `chunk_overlap` are measured in characters for the character-based splitters below. Token splitting measures tokens. Start with the recursive splitter for general prose, then tune based on retrieval results.

## 1. Install and import

Text splitters are distributed as a separate package. Install it into the same environment as your notebook kernel.

In [ ]:
%pip install langchain-text-splitters

In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    HTMLHeaderTextSplitter,
    Language,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    TokenTextSplitter,
)

text = """Retrieval augmented generation (RAG) grounds a language model response in retrieved information.

A document is usually too long to send as one piece. Chunking divides it into smaller passages.
Each passage can be embedded and retrieved independently, while overlap helps preserve context
that crosses a chunk boundary.

Chunk size is a tradeoff. Very small chunks can lose context; very large chunks can be less
focused and may exceed the model context window."""
print(f'{len(text)} characters')

461 characters


## 2. Fixed-size character chunks

`CharacterTextSplitter` splits on a chosen separator and combines pieces up to a size limit. This is simple, but may produce chunks that are shorter than the limit or awkwardly split when separators are sparse.

In [2]:
char_splitter = CharacterTextSplitter(
    separator=' ',
    chunk_size=120,
    chunk_overlap=20,
)
char_chunks = char_splitter.split_text(text)
for i, chunk in enumerate(char_chunks, start=1):
    print(f'Chunk {i} ({len(chunk)} chars): {chunk!r}')

Chunk 1 (119 chars): 'Retrieval augmented generation (RAG) grounds a language model response in retrieved information.\n\nA document is usually'
Chunk 2 (113 chars): 'document is usually too long to send as one piece. Chunking divides it into smaller passages.\nEach passage can be'
Chunk 3 (110 chars): 'passage can be embedded and retrieved independently, while overlap helps preserve context\nthat crosses a chunk'
Chunk 4 (113 chars): 'crosses a chunk boundary.\n\nChunk size is a tradeoff. Very small chunks can lose context; very large chunks can be'
Chunk 5 (73 chars): 'large chunks can be less\nfocused and may exceed the model context window.'


## 3. Recursive splitting for general text

`RecursiveCharacterTextSplitter` tries paragraph breaks first, then line breaks, spaces, and finally individual characters. This usually keeps related text together better than a single fixed separator.

In [3]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    length_function=len,
    add_start_index=True,
)
recursive_chunks = recursive_splitter.split_text(text)
for i, chunk in enumerate(recursive_chunks, start=1):
    print(f'Chunk {i} ({len(chunk)} chars): {chunk!r}')

Chunk 1 (96 chars): 'Retrieval augmented generation (RAG) grounds a language model response in retrieved information.'
Chunk 2 (95 chars): 'A document is usually too long to send as one piece. Chunking divides it into smaller passages.'
Chunk 3 (125 chars): 'Each passage can be embedded and retrieved independently, while overlap helps preserve context\nthat crosses a chunk boundary.'
Chunk 4 (140 chars): 'Chunk size is a tradeoff. Very small chunks can lose context; very large chunks can be less\nfocused and may exceed the model context window.'


## 4. Split `Document` objects and keep metadata

Use `split_documents` when your loader already returned LangChain documents. The splitter carries each source document's metadata to its chunks and adds a start index when requested.

In [4]:
source_doc = Document(
    page_content=text,
    metadata={'source': 'rag_intro.txt', 'topic': 'RAG'},
)
doc_chunks = recursive_splitter.split_documents([source_doc])
print(f'Created {len(doc_chunks)} chunks')
print(doc_chunks[0].page_content)
print(doc_chunks[0].metadata)

Created 4 chunks
Retrieval augmented generation (RAG) grounds a language model response in retrieved information.
{'source': 'rag_intro.txt', 'topic': 'RAG', 'start_index': 0}


## 5. Markdown header splitting

`MarkdownHeaderTextSplitter` groups Markdown content under headings and records heading paths in metadata. It is useful when section boundaries carry meaning. It does not enforce a maximum chunk size, so you can pass its output through a recursive splitter afterward if needed.

In [5]:
markdown_text = """# Document Loaders
Loaders read files and create documents.

## Text files
TextLoader reads plain text.

## PDF files
PyPDFLoader can create one document per page.
"""
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[('#', 'header_1'), ('##', 'header_2')]
)
markdown_chunks = header_splitter.split_text(markdown_text)
for chunk in markdown_chunks:
    print(chunk.metadata, '=>', chunk.page_content)

{'header_1': 'Document Loaders'} => Loaders read files and create documents.
{'header_1': 'Document Loaders', 'header_2': 'Text files'} => TextLoader reads plain text.
{'header_1': 'Document Loaders', 'header_2': 'PDF files'} => PyPDFLoader can create one document per page.


## 6. HTML header splitting

`HTMLHeaderTextSplitter` separates content at HTML headings and stores the heading hierarchy in metadata. It parses HTML strings; use a web loader first if you need to fetch a URL.

In [9]:
html_text = """<html><body>
<h1>RAG guide</h1><p>RAG retrieves useful context for a response.</p>
<h2>Chunking</h2><p>Chunking divides a document into passages.</p>
</body></html>"""
html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[('h1', 'header_1'), ('h2', 'header_2')]
)
html_chunks = html_splitter.split_text(html_text)
for chunk in html_chunks:
    print(chunk.metadata, '=>', chunk.page_content)

{'header_1': 'RAG guide'} => RAG guide
{'header_1': 'RAG guide'} => RAG retrieves useful context for a response.
{'header_1': 'RAG guide', 'header_2': 'Chunking'} => Chunking
{'header_1': 'RAG guide', 'header_2': 'Chunking'} => Chunking divides a document into passages.


## 7. Code-aware splitting

`RecursiveCharacterTextSplitter.from_language` uses separators suited to a programming language, such as class and function boundaries. It helps keep code units together when possible but does not parse code into a syntax tree.

In [10]:
python_code = """class Retriever:
    def __init__(self, index):
        self.index = index

    def search(self, query):
        return self.index.similarity_search(query, k=3)

def format_context(documents):
    return "\n".join(doc.page_content for doc in documents)
"""
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=100,
    chunk_overlap=15,
)
code_chunks = python_splitter.split_text(python_code)
for i, chunk in enumerate(code_chunks, start=1):
    print(f'Chunk {i}:\n{chunk}\n')

Chunk 1:
class Retriever:
    def __init__(self, index):
        self.index = index

Chunk 2:
def search(self, query):
        return self.index.similarity_search(query, k=3)

Chunk 3:
def format_context(documents):
    return "
".join(doc.page_content for doc in documents)



## 8. Token-based splitting

`TokenTextSplitter` limits chunks by tokenizer tokens instead of characters. This example uses the `cl100k_base` encoding and may require `tiktoken` (`%pip install tiktoken`). Token counts vary by encoding and model.

In [ ]:
# Install the optional tokenizer dependency first if needed:
# %pip install tiktoken
# token_splitter = TokenTextSplitter(
#     encoding_name='cl100k_base',
#     chunk_size=80,
#     chunk_overlap=10,
# )
# token_chunks = token_splitter.split_text(text)
# print(token_chunks[0])

## 9. Compare overlap and inspect results

Overlap repeats a small amount of text at the edge of neighboring chunks. Inspect actual chunks and their metadata before indexing; chunk size is a starting point, not a universal setting.

In [6]:
for i, chunk in enumerate(doc_chunks, start=1):
    print(f'{i}: {chunk.metadata} | {chunk.page_content!r}')

1: {'source': 'rag_intro.txt', 'topic': 'RAG', 'start_index': 0} | 'Retrieval augmented generation (RAG) grounds a language model response in retrieved information.'
2: {'source': 'rag_intro.txt', 'topic': 'RAG', 'start_index': 98} | 'A document is usually too long to send as one piece. Chunking divides it into smaller passages.'
3: {'source': 'rag_intro.txt', 'topic': 'RAG', 'start_index': 194} | 'Each passage can be embedded and retrieved independently, while overlap helps preserve context\nthat crosses a chunk boundary.'
4: {'source': 'rag_intro.txt', 'topic': 'RAG', 'start_index': 321} | 'Chunk size is a tradeoff. Very small chunks can lose context; very large chunks can be less\nfocused and may exceed the model context window.'
